In [ ]:

# ASSIGNMENT 4 – CO4
from pathlib import Path
import subprocess
import zipfile

PROJECT = Path("/content/CSA09-Assignment4")
PROJECT.mkdir(parents=True, exist_ok=True)

def write_file(relative_path, content):
    path = PROJECT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    return path


In [ ]:
write_file('Task01_FeePayment/FeeWindow.java', 'import javax.swing.*;\nimport java.awt.*;\nimport java.awt.event.*;\nimport java.math.BigDecimal;\n \npublic class FeeWindow extends JFrame {\n    private final JTextField regNo = new JTextField(14);\n    private final JTextField amount = new JTextField(10);\n    private final JLabel status = new JLabel("Enter details to pay.");\n    private final FeeService service = new FeeService();\n \n    public FeeWindow() {\n        super("Student Fee Payment");\n        setLayout(new BorderLayout(8, 8));\n        JPanel form = new JPanel(new GridLayout(2, 2, 6, 6));\n        form.add(new JLabel("Registration No:")); form.add(regNo);\n        form.add(new JLabel("Amount (INR):")); form.add(amount);\n        JButton pay = new JButton("Pay");\n        JButton clear = new JButton("Clear");\n        JButton receipt = new JButton("Receipt");\n        JPanel actions = new JPanel(new FlowLayout());\n        actions.add(pay); actions.add(clear); actions.add(receipt);\n        add(form, BorderLayout.CENTER); add(actions, BorderLayout.SOUTH);\n        add(status, BorderLayout.NORTH);\n \n        pay.addActionListener(e -> payFee());\n        clear.addActionListener(e -> { regNo.setText(""); amount.setText(""); status.setText("Cleared."); });\n        receipt.addActionListener(e -> JOptionPane.showMessageDialog(this,\n                service.getLastReceipt(), "Receipt", JOptionPane.INFORMATION_MESSAGE));\n        addWindowListener(new WindowAdapter() {\n            @Override public void windowClosing(WindowEvent e) { dispose(); }\n        });\n        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE);\n        pack(); setLocationRelativeTo(null);\n    }\n \n    private void payFee() {\n        String id = regNo.getText().trim();\n        final BigDecimal value;\n        try {\n            value = new BigDecimal(amount.getText().trim());\n            if (id.isEmpty() || value.signum() <= 0) throw new IllegalArgumentException();\n        } catch (Exception ex) {\n            JOptionPane.showMessageDialog(this, "Enter a registration number and positive amount.",\n                    "Invalid input", JOptionPane.ERROR_MESSAGE);\n            return;\n        }\n        // Service/database work is delegated to a background worker.\n        new SwingWorker<String, Void>() {\n            protected String doInBackground() throws Exception {\n                return service.pay(id, value);\n            }\n            protected void done() {\n                try { status.setText(get()); }\n                catch (Exception ex) {\n                    status.setText("Payment failed.");\n                    JOptionPane.showMessageDialog(FeeWindow.this, ex.getMessage(),\n                            "Payment error", JOptionPane.ERROR_MESSAGE);\n                }\n            }\n        }.execute();\n    }\n \n    public static void main(String[] args) {\n        SwingUtilities.invokeLater(() -> new FeeWindow().setVisible(true));\n    }\n}')

In [ ]:
write_file('Task01_FeePayment/FeeService.java', 'import java.math.BigDecimal;\nclass FeeService {\n    private final FeeDAO dao = new FeeDAO();\n    private String lastReceipt = "No payment yet.";\n    String pay(String regNo, BigDecimal amount) throws Exception {\n        if (regNo == null || regNo.trim().isEmpty())\n            throw new IllegalArgumentException("Registration number is required.");\n        if (amount == null || amount.signum() <= 0)\n            throw new IllegalArgumentException("Amount must be positive.");\n        long id = dao.insertPayment(regNo.trim(), amount);\n        lastReceipt = "Receipt / Transaction: " + id + "\\nRegistration: " + regNo\n                + "\\nAmount: Rs. " + amount;\n        return "Payment saved. Transaction ID: " + id;\n    }\n    String getLastReceipt() { return lastReceipt; }\n}')

In [ ]:
write_file('Task01_FeePayment/FeeDAO.java', 'import java.sql.*;\nimport java.math.BigDecimal;\n\nimport java.math.BigDecimal;\nclass FeeDAO {\n    private static final String URL = "jdbc:mysql://localhost:3306/campus";\n    private static final String USER = "root";\n    private static final String PWD = "CHANGE_ME";\n    long insertPayment(String regNo, BigDecimal amount) throws SQLException {\n        String sql = "INSERT INTO fees(reg_no, amount) VALUES (?, ?)";\n        try (Connection c = DriverManager.getConnection(URL, USER, PWD);\n             PreparedStatement ps = c.prepareStatement(sql, Statement.RETURN_GENERATED_KEYS)) {\n            ps.setString(1, regNo); ps.setBigDecimal(2, amount);\n            ps.executeUpdate();\n            try (ResultSet rs = ps.getGeneratedKeys()) {\n                if (rs.next()) return rs.getLong(1);\n            }\n        }\n        throw new SQLException("Payment was not assigned a transaction ID.");\n    }\n}')

In [ ]:
write_file('Task02_AppointmentDesk/AppointmentDesk.java', 'import javax.swing.*;\nimport java.awt.*;\nimport java.awt.event.*;\nimport java.util.*;\n\npublic class AppointmentDesk extends JFrame {\n    private final JTextField patientId = new JTextField(12);\n    private final JComboBox<String> departmentCombo =\n            new JComboBox<>(new String[]{"Cardiology", "Neurology", "Orthopedics"});\n    private final JComboBox<Doctor> doctorCombo = new JComboBox<>();\n    private final JComboBox<String> timeCombo =\n            new JComboBox<>(new String[]{"09:00 AM", "10:00 AM", "11:00 AM"});\n    private final JLabel status = new JLabel("Enter patient details.");\n    private boolean bookingInProgress = false;\n\n    private final Map<String, java.util.List<Doctor>> doctorMap = Map.of(\n        "Cardiology", java.util.List.of(new Doctor("Dr. Anand"), new Doctor("Dr. Priya")),\n        "Neurology", java.util.List.of(new Doctor("Dr. Kumar"), new Doctor("Dr. Meena")),\n        "Orthopedics", java.util.List.of(new Doctor("Dr. Ravi"), new Doctor("Dr. Sara"))\n    );\n\n    public AppointmentDesk() {\n        super("Hospital Appointment Desk");\n        setLayout(new BorderLayout(10, 10));\n\n        JPanel patientPanel = new JPanel(new GridLayout(1, 2, 6, 6));\n        patientPanel.add(new JLabel("Patient ID"));\n        patientPanel.add(patientId);\n\n        JPanel appointmentPanel = new JPanel(new GridLayout(3, 2, 6, 6));\n        appointmentPanel.add(new JLabel("Department"));\n        appointmentPanel.add(departmentCombo);\n        appointmentPanel.add(new JLabel("Doctor"));\n        appointmentPanel.add(doctorCombo);\n        appointmentPanel.add(new JLabel("Time slot"));\n        appointmentPanel.add(timeCombo);\n\n        JButton bookButton = new JButton("Book Appointment");\n        JButton clearButton = new JButton("Clear");\n        JPanel actionPanel = new JPanel(new FlowLayout());\n        actionPanel.add(bookButton);\n        actionPanel.add(clearButton);\n\n        JPanel top = new JPanel(new BorderLayout(8, 8));\n        top.add(patientPanel, BorderLayout.NORTH);\n        top.add(appointmentPanel, BorderLayout.CENTER);\n        add(top, BorderLayout.CENTER);\n        add(actionPanel, BorderLayout.SOUTH);\n        add(status, BorderLayout.NORTH);\n\n        departmentCombo.addItemListener(e -> {\n            if (e.getStateChange() == ItemEvent.SELECTED) {\n                String department = (String)e.getItem();\n                doctorCombo.removeAllItems();\n                for (Doctor doctor : doctorMap.getOrDefault(department, Collections.emptyList())) {\n                    doctorCombo.addItem(doctor);\n                }\n            }\n        });\n        bookButton.addActionListener(e -> bookAppointment());\n        clearButton.addActionListener(e -> {\n            patientId.setText("");\n            departmentCombo.setSelectedIndex(0);\n            timeCombo.setSelectedIndex(0);\n            status.setText("Form cleared.");\n        });\n\n        addWindowListener(new WindowAdapter() {\n            @Override public void windowClosing(WindowEvent e) {\n                if (bookingInProgress) {\n                    int choice = JOptionPane.showConfirmDialog(AppointmentDesk.this,\n                        "A booking is in progress. Close anyway?", "Confirm close",\n                        JOptionPane.YES_NO_OPTION);\n                    if (choice == JOptionPane.YES_OPTION) dispose();\n                } else dispose();\n            }\n        });\n        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE);\n        departmentCombo.setSelectedIndex(0);\n        setSize(560, 280);\n        setLocationRelativeTo(null);\n    }\n\n    private void bookAppointment() {\n        String id = patientId.getText().trim();\n        Doctor doctor = (Doctor)doctorCombo.getSelectedItem();\n        if (id.isEmpty()) {\n            JOptionPane.showMessageDialog(this, "Patient ID is required.");\n            return;\n        }\n        if (doctor == null || timeCombo.getSelectedItem() == null) {\n            JOptionPane.showMessageDialog(this, "Select a doctor and time slot.");\n            return;\n        }\n        bookingInProgress = true;\n        status.setText("Booking appointment...");\n        // Demonstration booking; replace this section with a service/database call.\n        bookingInProgress = false;\n        status.setText("Appointment booked for " + id + ".");\n        JOptionPane.showMessageDialog(this, "Appointment confirmed with " + doctor);\n    }\n\n    static class Doctor {\n        final String name;\n        Doctor(String name) { this.name = name; }\n        public String toString() { return name; }\n    }\n\n    public static void main(String[] args) {\n        SwingUtilities.invokeLater(() -> new AppointmentDesk().setVisible(true));\n    }\n}')

In [ ]:
write_file('Task03_LibrarySearch/LibrarySearchFrame.java', 'import javax.swing.*;\nimport javax.swing.table.DefaultTableModel;\nimport java.awt.*;\nimport java.util.ArrayList;\nimport java.util.List;\n \npublic class LibrarySearchFrame extends JFrame {\n    private final JTextField query = new JTextField(24);\n    private final JComboBox<String> category =\n            new JComboBox<>(new String[]{"All", "Title", "Author", "Subject"});\n    private final DefaultTableModel model = new DefaultTableModel(\n            new String[]{"Book ID", "Title", "Author", "Category", "Status"}, 0);\n    private final JTable results = new JTable(model);\n    private final JLabel status = new JLabel("Enter a search term.");\n    private int page = 0;\n \n    public LibrarySearchFrame() {\n        super("Campus Library Search");\n        setLayout(new BorderLayout(8, 8));\n \n        JPanel searchPanel = new JPanel(new FlowLayout(FlowLayout.LEFT));\n        searchPanel.add(new JLabel("Search:"));\n        searchPanel.add(query); searchPanel.add(category);\n        JButton search = new JButton("Search");\n        searchPanel.add(search);\n        add(searchPanel, BorderLayout.NORTH);\n \n        JPanel resultPanel = new JPanel(new BorderLayout());\n        results.setFillsViewportHeight(true);\n        resultPanel.add(new JScrollPane(results), BorderLayout.CENTER);\n        add(resultPanel, BorderLayout.CENTER);\n \n        JButton prev = new JButton("Prev");\n        JButton next = new JButton("Next");\n        JButton reserve = new JButton("Reserve");\n        JPanel controls = new JPanel(new FlowLayout(FlowLayout.LEFT));\n        controls.add(prev); controls.add(next); controls.add(reserve);\n        controls.add(status);\n        add(controls, BorderLayout.SOUTH);\n \n        search.addActionListener(e -> performSearch());\n        query.addActionListener(e -> performSearch());\n        prev.addActionListener(e -> { if (page > 0) page--; performSearch(); });\n        next.addActionListener(e -> { page++; performSearch(); });\n        reserve.addActionListener(e -> reserveSelected());\n        setDefaultCloseOperation(EXIT_ON_CLOSE);\n        setMinimumSize(new Dimension(650, 400));\n        setSize(850, 550);\n        setLocationRelativeTo(null);\n        loadDemoRows();\n    }\n \n    private void loadDemoRows() {\n        model.addRow(new Object[]{"B101","Java Programming","Herbert Schildt","Programming","Available"});\n        model.addRow(new Object[]{"B102","Database Systems","Elmasri","Database","Issued"});\n        model.addRow(new Object[]{"B103","Event-Driven Design","Deitel","Programming","Available"});\n        status.setText("3 sample records.");\n    }\n \n    private void performSearch() {\n        String text = query.getText().trim();\n        status.setText("Searching for: " + text + " | Page " + (page + 1));\n        // Replace this demo filter with DAO search; run database work in SwingWorker.\n    }\n \n    private void reserveSelected() {\n        int row = results.getSelectedRow();\n        if (row < 0) {\n            JOptionPane.showMessageDialog(this, "Select a book first.");\n            return;\n        }\n        JOptionPane.showMessageDialog(this, "Reservation request for " +\n                model.getValueAt(row, 1) + " recorded.");\n    }\n \n    public static void main(String[] args) {\n        SwingUtilities.invokeLater(() -> new LibrarySearchFrame().setVisible(true));\n    }\n}')

In [ ]:
write_file('Task04_DrawPad/DrawPad.java', 'import java.awt.*;\nimport java.awt.event.*;\nimport java.util.ArrayList;\nimport java.util.List;\n \npublic class DrawPad extends Frame {\n    private final List<Point> points = new ArrayList<>();\n    private final Label status = new Label("Clicks: 0");\n \n    public DrawPad() {\n        super("Draw Pad");\n        add(status, BorderLayout.SOUTH);\n        addMouseListener(new MouseAdapter() {\n            @Override public void mouseClicked(MouseEvent e) {\n                points.add(e.getPoint());\n                status.setText("Clicks: " + points.size());\n                repaint();\n            }\n        });\n        addWindowListener(new WindowAdapter() {\n            @Override public void windowClosing(WindowEvent e) { dispose(); }\n        });\n        setSize(400, 300);\n    }\n \n    @Override public void paint(Graphics g) {\n        super.paint(g);\n        for (Point point : points)\n            g.fillOval(point.x - 3, point.y - 3, 6, 6);\n    }\n \n    public static void main(String[] args) {\n        EventQueue.invokeLater(() -> new DrawPad().setVisible(true));\n    }\n}')

In [ ]:
write_file('Task05_CurrencyConverter/ConverterFrameListener.java', 'import javax.swing.*;\nimport java.awt.*;\nimport java.awt.event.*;\nimport java.text.DecimalFormat;\n \npublic class ConverterFrameListener extends JFrame implements ActionListener {\n    JTextField amount = new JTextField(10);\n    JComboBox<String> target = new JComboBox<>(new String[]{"USD","EUR"});\n    JButton button = new JButton("Convert");\n    double rate(String c) { return c.equals("USD") ? .012 : .011; }\n \n    ConverterFrameListener() {\n        setTitle("Converter – Frame Listener"); setLayout(new FlowLayout());\n        add(new JLabel("INR")); add(amount); add(target); add(button);\n        button.addActionListener(this); amount.addActionListener(this);\n        addWindowListener(new WindowAdapter() {\n            @Override public void windowClosing(WindowEvent e) { dispose(); }\n        });\n        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE); pack();\n    }\n    public void actionPerformed(ActionEvent e) { convert(); }\n    private void convert() {\n        try {\n            double value = Double.parseDouble(amount.getText().trim());\n            if (!Double.isFinite(value) || value < 0) throw new NumberFormatException();\n            String c = (String) target.getSelectedItem();\n            JOptionPane.showMessageDialog(this,\n                new DecimalFormat("0.00").format(value * rate(c)) + " " + c);\n        } catch (NumberFormatException ex) {\n            JOptionPane.showMessageDialog(this, "Invalid amount", "Error", JOptionPane.ERROR_MESSAGE);\n        }\n    }\n    public static void main(String[] a) { SwingUtilities.invokeLater(() -> new ConverterFrameListener().setVisible(true)); }\n}')

In [ ]:
write_file('Task05_CurrencyConverter/ConverterNamedInner.java', 'import javax.swing.*;\nimport java.awt.*;\nimport java.awt.event.*;\n\nclass ConverterNamedInner extends JFrame {\n    private final JTextField amount = new JTextField(10);\n    private final JComboBox<String> target = new JComboBox<>(new String[]{"USD","EUR"});\n    ConverterNamedInner() {\n        setLayout(new FlowLayout()); JButton button = new JButton("Convert");\n        add(amount); add(target); add(button);\n        ConvertHandler handler = new ConvertHandler();\n        button.addActionListener(handler); amount.addActionListener(handler);\n        addWindowListener(new WindowAdapter() {\n            @Override public void windowClosing(WindowEvent e) { dispose(); }\n        });\n        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE); pack();\n    }\n    private class ConvertHandler implements ActionListener {\n        public void actionPerformed(ActionEvent e) { convert(); }\n    }\n    private void convert() {\n        try {\n            double n = Double.parseDouble(amount.getText().trim());\n            if (!Double.isFinite(n) || n < 0) throw new NumberFormatException();\n            String c=(String)target.getSelectedItem(); double r=c.equals("USD")?.012:.011;\n            JOptionPane.showMessageDialog(this,String.format("%.2f %s",n*r,c));\n        } catch (NumberFormatException ex) {\n            JOptionPane.showMessageDialog(this,"Invalid amount","Error",JOptionPane.ERROR_MESSAGE);\n        }\n    }\n\n    public static void main(String[] args) {\n        SwingUtilities.invokeLater(() -> new ConverterNamedInner().setVisible(true));\n    }\n}\n')

In [ ]:
write_file('Task05_CurrencyConverter/ConverterAnonymous.java', 'import javax.swing.*;\nimport java.awt.*;\nimport java.awt.event.*;\n\nclass ConverterAnonymous extends JFrame {\n    private final JTextField amount = new JTextField(10);\n    private final JComboBox<String> target = new JComboBox<>(new String[]{"USD","EUR"});\n    ConverterAnonymous() {\n        setLayout(new FlowLayout()); JButton button = new JButton("Convert");\n        add(amount); add(target); add(button);\n        ActionListener handler = new ActionListener() {\n            @Override public void actionPerformed(ActionEvent e) { convert(); }\n        };\n        button.addActionListener(handler); amount.addActionListener(handler);\n        addWindowListener(new WindowAdapter() {\n            @Override public void windowClosing(WindowEvent e) { dispose(); }\n        });\n        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE); pack();\n    }\n    private void convert() {\n        try {\n            double n=Double.parseDouble(amount.getText().trim());\n            if (!Double.isFinite(n) || n < 0) throw new NumberFormatException();\n            String c=(String)target.getSelectedItem();\n            JOptionPane.showMessageDialog(this,String.format("%.2f %s",n*(c.equals("USD")?.012:.011),c));\n        } catch (NumberFormatException ex) {\n            JOptionPane.showMessageDialog(this,"Invalid amount","Error",JOptionPane.ERROR_MESSAGE);\n        }\n    }\n\n    public static void main(String[] args) {\n        SwingUtilities.invokeLater(() -> new ConverterAnonymous().setVisible(true));\n    }\n}\n')

In [ ]:
write_file('Task05_CurrencyConverter/Converter.java', 'import java.awt.FlowLayout;\nimport java.awt.event.*;\nimport javax.swing.*;\n \npublic class Converter extends JFrame {\n    private final JTextField amount = new JTextField(10);\n    private final JComboBox<String> target = new JComboBox<>(new String[]{"USD", "EUR"});\n \n    private static double rate(String currency) {\n        return currency.equals("USD") ? 0.012 : 0.011;\n    }\n \n    public Converter() {\n        super("Currency Converter");\n        setLayout(new FlowLayout());\n        JButton convertButton = new JButton("Convert");\n        add(new JLabel("INR")); add(amount); add(target);\n        add(convertButton);\n        ActionListener convertAction = e -> convert(); // one lambda listener\n        convertButton.addActionListener(convertAction);\n        amount.addActionListener(convertAction);       // Enter uses same listener\n        addWindowListener(new WindowAdapter() {\n            @Override public void windowClosing(WindowEvent e) { dispose(); }\n        });\n        setDefaultCloseOperation(DO_NOTHING_ON_CLOSE);\n        pack(); setLocationRelativeTo(null);\n    }\n \n    private void convert() {\n        try {\n            double value = Double.parseDouble(amount.getText().trim());\n            if (!Double.isFinite(value) || value < 0) throw new NumberFormatException();\n            String currency = (String) target.getSelectedItem();\n            String output = String.format("%.2f %s", value * rate(currency), currency);\n            JOptionPane.showMessageDialog(this, output, "Converted amount",\n                    JOptionPane.INFORMATION_MESSAGE);\n        } catch (NumberFormatException ex) {\n            JOptionPane.showMessageDialog(this, "Invalid amount", "Input error",\n                    JOptionPane.ERROR_MESSAGE);\n        }\n    }\n \n    public static void main(String[] args) {\n        SwingUtilities.invokeLater(() -> new Converter().setVisible(true));\n    }\n}')

In [ ]:
write_file('Task06_EventRegistration/schema.sql', "CREATE DATABASE IF NOT EXISTS campus_events;\nUSE campus_events;\n \nCREATE TABLE IF NOT EXISTS events (\n    event_id INT PRIMARY KEY AUTO_INCREMENT,\n    event_name VARCHAR(120) NOT NULL UNIQUE,\n    event_date DATE NOT NULL,\n    capacity INT NOT NULL DEFAULT 100,\n    active BOOLEAN NOT NULL DEFAULT TRUE,\n    CHECK (capacity > 0)\n);\n \nCREATE TABLE IF NOT EXISTS registrations (\n    registration_id BIGINT PRIMARY KEY AUTO_INCREMENT,\n    student_id VARCHAR(30) NOT NULL,\n    student_name VARCHAR(100) NOT NULL,\n    email VARCHAR(150) NOT NULL,\n    event_id INT NOT NULL,\n    registered_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    CONSTRAINT fk_registration_event\n        FOREIGN KEY (event_id) REFERENCES events(event_id),\n    CONSTRAINT uq_student_event UNIQUE (student_id, event_id)\n);\n \nCREATE INDEX idx_registrations_event\n    ON registrations(event_id, registered_at);\n \nINSERT INTO events(event_name, event_date, capacity) VALUES\n('Cultural Fest', '2026-12-10', 500),\n('Tech Symposium', '2026-12-15', 300),\n('Music Night', '2026-12-20', 250);")

In [ ]:
write_file('Task06_EventRegistration/RegistrationDAO.java', 'import java.sql.*;\nimport java.util.*;\n \npublic class RegistrationDAO {\n    private static final String URL = "jdbc:mysql://localhost:3306/campus_events";\n    private static final String USER = "root";\n    private static final String PASSWORD = "CHANGE_ME";\n \n    public void insert(Registration r) throws SQLException {\n        String sql = "INSERT INTO registrations(student_id,student_name,email,event_id) VALUES(?,?,?,?)";\n        try (Connection c = DriverManager.getConnection(URL, USER, PASSWORD);\n             PreparedStatement ps = c.prepareStatement(sql)) {\n            ps.setString(1, r.studentId);\n            ps.setString(2, r.studentName);\n            ps.setString(3, r.email);\n            ps.setInt(4, r.eventId);\n            ps.executeUpdate();\n        }\n    }\n \n    public List<Registration> fetchPage(Integer eventId, int limit, int offset)\n            throws SQLException {\n        String base = "SELECT r.registration_id,r.student_id,r.student_name,r.email," +\n                "r.event_id,e.event_name,r.registered_at FROM registrations r " +\n                "JOIN events e ON e.event_id=r.event_id ";\n        String where = eventId == null ? "" : "WHERE r.event_id=? ";\n        String sql = base + where + "ORDER BY r.registered_at DESC LIMIT ? OFFSET ?";\n        List<Registration> list = new ArrayList<>();\n        try (Connection c = DriverManager.getConnection(URL, USER, PASSWORD);\n             PreparedStatement ps = c.prepareStatement(sql)) {\n            int i = 1;\n            if (eventId != null) ps.setInt(i++, eventId);\n            ps.setInt(i++, limit); ps.setInt(i, offset);\n            try (ResultSet rs = ps.executeQuery()) {\n                while (rs.next()) {\n                    list.add(new Registration(rs.getLong("registration_id"),\n                        rs.getString("student_id"), rs.getString("student_name"),\n                        rs.getString("email"), rs.getInt("event_id"),\n                        rs.getString("event_name"), rs.getTimestamp("registered_at")));\n                }\n            }\n        }\n        return list;\n    }\n \n    public int count(Integer eventId) throws SQLException {\n        String sql = "SELECT COUNT(*) FROM registrations" +\n                (eventId == null ? "" : " WHERE event_id=?");\n        try (Connection c = DriverManager.getConnection(URL, USER, PASSWORD);\n             PreparedStatement ps = c.prepareStatement(sql)) {\n            if (eventId != null) ps.setInt(1, eventId);\n            try (ResultSet rs = ps.executeQuery()) {\n                rs.next(); return rs.getInt(1);\n            }\n        }\n    }\n}')

In [ ]:
write_file('Task06_EventRegistration/Registration.java', 'import java.sql.Timestamp;\nclass Registration {\n    final long id; final String studentId, studentName, email, eventName;\n    final int eventId; final Timestamp registeredAt;\n \n    Registration(String studentId, String studentName, String email, int eventId) {\n        this(0, studentId, studentName, email, eventId, "", null);\n    }\n    Registration(long id, String studentId, String studentName, String email,\n                 int eventId, String eventName, Timestamp registeredAt) {\n        this.id=id; this.studentId=studentId; this.studentName=studentName;\n        this.email=email; this.eventId=eventId; this.eventName=eventName;\n        this.registeredAt=registeredAt;\n    }\n}\n \nclass RegistrationService {\n    private final RegistrationDAO dao = new RegistrationDAO();\n    void register(String id, String name, String email, int eventId) throws Exception {\n        if (id == null || id.trim().isEmpty()) throw new IllegalArgumentException("Student ID is required.");\n        if (name == null || name.trim().isEmpty()) throw new IllegalArgumentException("Student name is required.");\n        if (email == null || !email.trim().matches("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+$"))\n            throw new IllegalArgumentException("Enter a valid email.");\n        if (eventId <= 0) throw new IllegalArgumentException("Select an event.");\n        dao.insert(new Registration(id.trim(), name.trim(), email.trim(), eventId));\n    }\n}')

In [ ]:
write_file('Task06_EventRegistration/EventRegistrationFrame.java', 'import javax.swing.*;\nimport javax.swing.table.DefaultTableModel;\nimport java.awt.*;\nimport java.awt.event.*;\nimport java.io.*;\nimport java.nio.charset.StandardCharsets;\nimport java.nio.file.*;\nimport java.sql.SQLException;\nimport java.util.List;\n \npublic class EventRegistrationFrame extends JFrame {\n    private final JTextField studentId = new JTextField(10);\n    private final JTextField studentName = new JTextField(14);\n    private final JTextField email = new JTextField(16);\n    private final JComboBox<EventItem> eventBox = new JComboBox<>(new EventItem[]{\n        new EventItem(1, "Cultural Fest"), new EventItem(2, "Tech Symposium"),\n        new EventItem(3, "Music Night")\n    });\n    private final JComboBox<EventItem> filterBox = new JComboBox<>();\n    private final DefaultTableModel model = new DefaultTableModel(\n        new String[]{"ID","Student ID","Name","Email","Event","Registered At"},0);\n    private final JTable table = new JTable(model);\n    private final JLabel status = new JLabel("Ready");\n    private final RegistrationService service = new RegistrationService();\n    private final RegistrationDAO dao = new RegistrationDAO();\n    private int page = 0;\n    private static final int PAGE_SIZE = 25;\n \n    public EventRegistrationFrame() {\n        super("Campus Event Registration");\n        setLayout(new BorderLayout(8,8));\n        JMenuBar bar = new JMenuBar();\n        JMenu file = new JMenu("File");\n        JMenuItem export = new JMenuItem("Export CSV");\n        JMenuItem exit = new JMenuItem("Exit");\n        file.add(export); file.add(exit); bar.add(file);\n        JMenu view = new JMenu("View");\n        JMenuItem refresh = new JMenuItem("Refresh");\n        view.add(refresh); bar.add(view); setJMenuBar(bar);\n        export.addActionListener(e -> exportCsv());\n        exit.addActionListener(e -> dispose());\n        refresh.addActionListener(e -> loadPage());\n \n        JPanel form = new JPanel(new FlowLayout(FlowLayout.LEFT));\n        form.setBorder(BorderFactory.createTitledBorder("Student Registration"));\n        form.add(new JLabel("Student ID")); form.add(studentId);\n        form.add(new JLabel("Name")); form.add(studentName);\n        form.add(new JLabel("Email")); form.add(email);\n        form.add(new JLabel("Event")); form.add(eventBox);\n        JButton register = new JButton("Register");\n        form.add(register);\n        add(form, BorderLayout.NORTH);\n \n        JPanel listing = new JPanel(new BorderLayout(5,5));\n        JPanel filter = new JPanel(new FlowLayout(FlowLayout.LEFT));\n        filter.add(new JLabel("Filter event:"));\n        filterBox.addItem(new EventItem(0, "All events"));\n        filterBox.addItem(new EventItem(1, "Cultural Fest"));\n        filterBox.addItem(new EventItem(2, "Tech Symposium"));\n        filterBox.addItem(new EventItem(3, "Music Night"));\n        JButton apply = new JButton("Apply");\n        JButton prev = new JButton("Previous");\n        JButton next = new JButton("Next");\n        filter.add(filterBox); filter.add(apply); filter.add(prev); filter.add(next);\n        listing.add(filter, BorderLayout.NORTH);\n        listing.add(new JScrollPane(table), BorderLayout.CENTER);\n        add(listing, BorderLayout.CENTER);\n        add(status, BorderLayout.SOUTH);\n \n        register.addActionListener(e -> registerStudent());\n        apply.addActionListener(e -> { page=0; loadPage(); });\n        prev.addActionListener(e -> { if(page>0) page--; loadPage(); });\n        next.addActionListener(e -> { page++; loadPage(); });\n \n        setDefaultCloseOperation(EXIT_ON_CLOSE);\n        setSize(1050,600); setLocationRelativeTo(null);\n        loadPage();\n    }\n \n    private Integer selectedFilter() {\n        EventItem item=(EventItem)filterBox.getSelectedItem();\n        return item==null || item.id==0 ? null : item.id;\n    }\n \n    private void registerStudent() {\n        EventItem event=(EventItem)eventBox.getSelectedItem();\n        if(event==null) { showError("Select an event."); return; }\n        setBusy(true);\n        new SwingWorker<Void,Void>() {\n            protected Void doInBackground() throws Exception {\n                service.register(studentId.getText(), studentName.getText(),\n                        email.getText(), event.id);\n                return null;\n            }\n            protected void done() {\n                setBusy(false);\n                try {\n                    get();\n                    JOptionPane.showMessageDialog(EventRegistrationFrame.this,\n                            "Registration successful.");\n                    loadPage();\n                } catch (Exception ex) {\n                    Throwable cause=ex.getCause();\n                    String msg=cause==null?ex.getMessage():cause.getMessage();\n                    if (cause instanceof SQLException &&\n                        "23000".equals(((SQLException)cause).getSQLState()))\n                        msg="Already registered for this event.";\n                    showError(msg);\n                }\n            }\n        }.execute();\n    }\n \n    private void loadPage() {\n        setBusy(true);\n        Integer eventId=selectedFilter();\n        int offset=page*PAGE_SIZE;\n        new SwingWorker<List<Registration>,Void>() {\n            protected List<Registration> doInBackground() throws Exception {\n                return dao.fetchPage(eventId,PAGE_SIZE,offset);\n            }\n            protected void done() {\n                setBusy(false);\n                try {\n                    List<Registration> rows=get();\n                    model.setRowCount(0);\n                    for(Registration r:rows)\n                        model.addRow(new Object[]{r.id,r.studentId,r.studentName,\n                            r.email,r.eventName,r.registeredAt});\n                    status.setText("Page "+(page+1)+" | Showing "+rows.size()+" rows");\n                } catch(Exception ex) { showError("Could not load registrations: "+ex.getMessage()); }\n            }\n        }.execute();\n    }\n \n    private void exportCsv() {\n        JFileChooser chooser=new JFileChooser();\n        if(chooser.showSaveDialog(this)!=JFileChooser.APPROVE_OPTION) return;\n        Path target=chooser.getSelectedFile().toPath();\n        Integer eventId=selectedFilter();\n        new SwingWorker<Void,Void>() {\n            protected Void doInBackground() throws Exception {\n                List<Registration> rows=dao.fetchPage(eventId,100000,0);\n                try(BufferedWriter w=Files.newBufferedWriter(target,StandardCharsets.UTF_8)) {\n                    w.write("Registration ID,Student ID,Name,Email,Event,Registered At\\n");\n                    for(Registration r:rows) {\n                        w.write(r.id+","+csv(r.studentId)+","+csv(r.studentName)+","+\n                            csv(r.email)+","+csv(r.eventName)+","+csv(String.valueOf(r.registeredAt))+"\\n");\n                    }\n                }\n                return null;\n            }\n            protected void done() {\n                try { get(); JOptionPane.showMessageDialog(EventRegistrationFrame.this,"CSV exported."); }\n                catch(Exception ex) { showError("Export failed: "+ex.getMessage()); }\n            }\n        }.execute();\n    }\n \n    private String csv(String value) {\n        String s=value==null?"":value.replace("\\"","\\"\\"");\n        return "\\""+s+"\\"";\n    }\n    private void setBusy(boolean busy) {\n        status.setText(busy?"Working...":"Ready");\n    }\n    private void showError(String message) {\n        JOptionPane.showMessageDialog(this,message,"Error",JOptionPane.ERROR_MESSAGE);\n    }\n \n    static class EventItem {\n        final int id; final String name;\n        EventItem(int id,String name){this.id=id;this.name=name;}\n        public String toString(){return name;}\n    }\n \n    public static void main(String[] args) {\n        SwingUtilities.invokeLater(() -> new EventRegistrationFrame().setVisible(true));\n    }\n}')

In [ ]:

TASKS = [
    "Task01_FeePayment", "Task02_AppointmentDesk", "Task03_LibrarySearch",
    "Task04_DrawPad", "Task05_CurrencyConverter", "Task06_EventRegistration"
]
for task in TASKS:
    java_files = sorted(str(p) for p in (PROJECT / task).glob("*.java"))
    result = subprocess.run(["javac"] + java_files, cwd=PROJECT,
                            capture_output=True, text=True)
    print(task, "COMPILED" if result.returncode == 0 else "COMPILATION ERROR")
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode: raise RuntimeError(f"Compilation failed in {task}")


In [ ]:

ZIP_PATH = Path("/content/CSA09-Assignment4-CO4.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob("*"):
        if path.is_file() and path.suffix != ".class":
            archive.write(path, path.relative_to(PROJECT.parent))
print(ZIP_PATH)
